In [1]:
import os
import numpy as np
import pandas as pd




In [2]:
def get_test_ids(test_path):
    """
    Return a list of BraTS21ID strings taken from the folder names
    inside the test directory.
    """
    ids = []
    for entry in sorted(os.scandir(test_path), key=lambda e: e.name):
        if entry.is_dir():
            ids.append(entry.name)  # folder name already zero‑padded, e.g. "00002"
    return ids




In [3]:
# possible locations for the test folder – include typical Kaggle and local paths
possible_paths = [
    "/kaggle/input/rsna-miccai-brain-tumor-radiogenomic-classification/test",
    "../input/rsna-miccai-brain-tumor-radiogenomic-classification/test",
    "input/rsna-miccai-brain-tumor-radiogenomic-classification/test",
    "./test",  # fallback to a local test folder if running locally
    "data/rsna-miccai-brain-tumor-radiogenomic-classification/test",
    "input/data/rsna-miccai-brain-tumor-radiogenomic-classification/test",
]
test_path = next((p for p in possible_paths if os.path.isdir(p)), None)
if test_path is None:
    raise FileNotFoundError(
        "Test directory not found in any of the expected locations."
    )

test_ids = get_test_ids(test_path)
if not test_ids:
    raise ValueError("No test IDs found – check the test directory structure.")

# locate the training labels CSV with similar fall‑backs
train_labels_path = next(
    (
        p
        for p in [
            "/kaggle/input/rsna-miccai-brain-tumor-radiogenomic-classification/train_labels.csv",
            "../input/rsna-miccai-brain-tumor-radiogenomic-classification/train_labels.csv",
            "input/rsna-miccai-brain-tumor-radiogenomic-classification/train_labels.csv",
            "./train_labels.csv",  # local fallback
            "data/rsna-miccai-brain-tumor-radiogenomic-classification/train_labels.csv",
            "input/data/rsna-miccai-brain-tumor-radiogenomic-classification/train_labels.csv",
        ]
        if os.path.isfile(p)
    ),
    None,
)

if train_labels_path is None:
    raise FileNotFoundError("train_labels.csv not found in any expected location.")
train_labels_df = pd.read_csv(train_labels_path)

# Exclude the three problematic subjects as advised in the competition note
problematic_ids = {"00109", "00123", "00709"}
train_labels_df = train_labels_df[~train_labels_df["BraTS21ID"].isin(problematic_ids)]

# Baseline prediction = overall mean MGMT_value, clipped to [0,1]
baseline_prob = float(np.clip(train_labels_df["MGMT_value"].mean(), 0.0, 1.0))

# Produce a constant prediction for every test case
predictions = np.full(len(test_ids), baseline_prob, dtype=float)



In [4]:
# Build the submission DataFrame with the required columns
submission_df = pd.DataFrame({"BraTS21ID": test_ids, "MGMT_value": predictions})

# Ensure the working directory exists and write the CSV
submission_dir = "/kaggle/working"
os.makedirs(submission_dir, exist_ok=True)
submission_path = os.path.join(submission_dir, "submission.csv")
submission_df.to_csv(submission_path, index=False)

print(f"Submission written to {os.path.abspath(submission_path)}")
print(f"Rows in submission: {submission_df.shape[0]}")

Submission written to /kaggle/working/submission.csv
Rows in submission: 60
